# Segmentasi per gigi — seed otomatis → SAM point-prompt

**Fokus tunggal notebook ini: setiap gigi tersegmentasi satu per satu, benar semua.**
Penomoran FDI dan metrik menyusul di notebook lain.

## Kenapa ganti pendekatan

Automatic mask generation menebar grid titik lalu berharap SAM menemukan sesuatu yang berguna.
Untuk foto intraoral itu gagal berulang kali — 6 mask untuk seluruh gambar, dan gigi berjejal
sering menyatu jadi satu blob.

Padahal kita tahu persis apa yang dicari: **12–20 gigi, tersusun sepanjang lengkung**. Kalau kita
bisa menaruh **satu titik di tengah tiap gigi**, SAM sangat andal mengubah titik itu jadi mask
presisi. Jadi persoalannya bukan "bagaimana mensegmentasi gigi", melainkan **"dari mana titiknya
datang"** — dan itu persoalan yang jauh lebih mudah.

```
foto
 └─► 1. region gigi        (ambang warna adaptif + morfologi)
      └─► 2. seed per gigi (distance transform + puncak lokal + lembah interproksimal)
           └─► 3. SAM point-prompt  (satu prompt per seed)
                └─► 4. split / merge  (perbaiki mask yang menggabung 2 gigi atau pecah)
                     └─► 5. quality gate  (tandai foto yang perlu koreksi)
                          └─► 6. editor klik  (perbaiki manual, seed diperbarui, ulangi 3-4)
```

## Kasus sulit yang ditangani

| Kasus | Penanganan |
|---|---|
| **Gigi bersentuhan** | Distance transform — gigi bersinggungan tetap punya inti sendiri |
| **Gigi tumpang tindih** | Region dipotong di garis interproksimal (tepi kuat) sebelum distance transform, lalu satu seed per komponen |
| **Gigi bawah tertutup gigi atas** | Region dipisah per lengkung dulu, seed dicari per lengkung |
| **Gigi pojok (posterior, gelap)** | Ambang Otsu per pita kolom, bukan global — sudut mulut lebih gelap |
| **Sela yang masih terlewat** | Lembah interproksimal dari proyeksi vertikal menambah seed |
| **Mask menggabung 2 gigi** | Dideteksi dari lebar berlebih, dipecah lewat watershed di dalam mask |

**Batas jujurnya:** uji sintetis menunjukkan jalur otomatis bertahan sampai sekitar **20% tumpang
tindih**. Di atas itu siluet menyatu dan garis interproksimal nyaris hilang — tidak ada penyetelan
parameter yang menolong, dan editor klik di Bagian 7 adalah jawaban yang benar. Untuk 18 foto,
beberapa klik jauh lebih murah daripada mengejar kesempurnaan otomatis.

## 0. Instalasi

```bash
pip install -U "transformers[torch]" scikit-image ipympl
```

- `scikit-image` untuk watershed & distance transform
- `ipympl` **opsional**, hanya untuk editor klik di Bagian 7

**Setelah `pip install ipympl`, kernel harus di-restart** — backend matplotlib hanya dibaca saat
kernel start. Tanpa restart, `%matplotlib widget` tetap gagal.

Notebook memeriksa ketersediaan ipympl lebih dulu dan mundur dengan rapi kalau tidak ada, jadi
Bagian 7 tidak akan meruntuhkan Run All. Kalau ipympl tetap rewel di VS Code (cukup umum),
pakai **Bagian 7b**: grid koordinat + ketik seed manual, tanpa dependensi apa pun.

In [ ]:
# ========= CONFIG =========
CFG = {
    "img_dir":  "Front Teeth drg Laura",
    "out_dir":  "seg_teeth",
    "model_id": "facebook/sam2.1-hiera-large",
    "force_cpu": False,
    "max_side": 1024,

    # --- 1. region gigi ---
    # "toothness" = value * (1 - saturation): tinggi utk gigi, rendah utk gusi/bibir.
    # Ambangnya OTSU, bukan persentil. Persentil selalu memilih fraksi tetap piksel —
    # di pita kolom tanpa gigi sekalipun ia tetap menandai 40% piksel sebagai "gigi".
    "col_tiles":   12,    # pita kolom utk koreksi lokal (sudut mulut lebih gelap)
    "local_floor": 0.70,  # ambang lokal tak boleh turun di bawah fraksi ini x ambang global
    "open_frac":   0.006, # jari-jari opening morfologi, fraksi lebar gambar
    "min_region":  0.02,  # buang komponen region gigi < fraksi ini dari region terbesar
    "max_region_frac": 0.55,  # region > ini dari gambar = ambang kelewat longgar

    # --- 2. seed ---
    # Jarak minimum antar seed diturunkan dari LEBAR GIGI itu sendiri: puncak distance
    # transform di dalam satu gigi kira-kira separuh lebarnya. Konstanta tetap menghasilkan
    # banyak seed per gigi kalau giginya besar.
    "seed_dist_mult":     0.90,   # x dt.max() -> jarak minimum antar seed
    "seed_min_dist_frac": 0.015,  # lantai, fraksi lebar gambar
    "seed_dt_frac":       0.35,   # ambang distance transform relatif thd maksimum
    "valley_prominence":  0.06,   # prominence lembah interproksimal (profil ternormalisasi)
    "edge_pct":           78,     # persentil tepi di dlm region -> dipotong (gigi tumpang tindih)
    "min_cut_frac":       0.22,   # komponen < fraksi ini x median dibuang (serpihan)
    "dedupe_mult":        0.55,   # x dt.max() utk melebur seed kembar (lebih kecil dr spacing)
    "max_seeds":          24,

    # --- 3. SAM ---
    "multimask": True,    # minta 3 kandidat per titik, pilih terbaik
    "sam_batch": 8,

    # --- 4. split / merge ---
    "split_wide_ratio": 1.55,  # mask > rasio ini x median lebar -> kandidat gabungan 2 gigi
    "merge_iou":        0.60,
    "min_area_frac":    0.0012,
    "max_area_frac":    0.10,
    "aspect_min":       0.35,
    "aspect_max":       5.00,

    # --- 5. quality gate ---
    "target_teeth":  (8, 20),   # rentang wajar jumlah gigi terlihat
    "min_coverage":  0.72,      # fraksi region gigi yg tertutup mask; di bawah ini -> ditandai
}
import os, json
os.makedirs(CFG["out_dir"], exist_ok=True)
print(json.dumps(CFG, indent=1))

## 1. Region gigi — ambang adaptif per kolom

In [ ]:
import glob, time, itertools, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, warnings
warnings.filterwarnings("ignore")
try:
    from pillow_heif import register_heif_opener; register_heif_opener()
except Exception: pass
from scipy import ndimage as ndi
from scipy.signal import find_peaks
from skimage.segmentation import watershed
from skimage.morphology import binary_opening, binary_closing, disk, remove_small_objects
from skimage.feature import peak_local_max

DEV = torch.device("cpu") if CFG["force_cpu"] else torch.device(
    "mps" if torch.backends.mps.is_available() else
    ("cuda" if torch.cuda.is_available() else "cpu"))
print("Device:", DEV)

EXTS = ("*.jpg","*.jpeg","*.JPG","*.JPEG","*.png","*.PNG","*.heic","*.HEIC")
paths = sorted(set(sum([glob.glob(os.path.join(CFG["img_dir"], e)) for e in EXTS], [])))
names = [os.path.splitext(os.path.basename(p))[0] for p in paths]
print(f"{len(paths)} foto")

def load_rgb(p):
    im = Image.open(p).convert("RGB")
    if max(im.size) > CFG["max_side"]:
        s = CFG["max_side"]/max(im.size)
        im = im.resize((int(im.width*s), int(im.height*s)), Image.LANCZOS)
    return np.asarray(im, np.uint8)

In [ ]:
def sat_val(rgb):
    a = rgb.astype(np.float32)/255.
    mx, mn = a.max(-1), a.min(-1)
    return np.where(mx > 1e-6, (mx-mn)/(mx+1e-6), 0.0), mx

def toothness(rgb):
    """Skor 'seperti gigi': terang DAN tidak jenuh. Gusi/bibir merah -> rendah."""
    sat, val = sat_val(rgb)
    return val * (1.0 - sat)

def tooth_region(rgb):
    """
    Region 'semua gigi', ambang OTSU pada skor toothness.

    Kenapa bukan persentil: persentil selalu memilih fraksi tetap piksel. Pada pita kolom
    yang isinya cuma gusi, `s <= percentile(s, 40)` tetap menandai 40% piksel sebagai gigi,
    dan region jadi penuh sampah. Otsu mencari lembah nyata antara dua modus, jadi kalau
    tidak ada gigi di pita itu, tidak ada yang dipilih (dijaga oleh pemeriksaan tile.max()).

    Koreksi per pita kolom hanya boleh MENURUNKAN ambang (sudut mulut lebih gelap), dibatasi
    `local_floor` supaya tidak menyeret gusi ikut masuk.
    """
    from skimage.filters import threshold_otsu
    H, W = rgb.shape[:2]
    t_map = toothness(rgb)
    try:
        t_glob = float(threshold_otsu(t_map))
    except Exception:
        t_glob = float(np.percentile(t_map, 75))
    reg = t_map > t_glob

    edges = np.linspace(0, W, max(3, CFG["col_tiles"])+1).astype(int)
    for x0, x1 in zip(edges[:-1], edges[1:]):
        tile = t_map[:, x0:x1]
        if tile.size < 50 or tile.max() < t_glob:
            continue                              # tidak ada gigi di pita ini
        try:
            t_loc = float(threshold_otsu(tile))
        except Exception:
            continue
        t = max(min(t_loc, t_glob), CFG["local_floor"]*t_glob)   # hanya boleh turun
        reg[:, x0:x1] = tile > t

    if reg.mean() > CFG["max_region_frac"]:       # ambang kelewat longgar -> perketat
        reg = t_map > float(np.percentile(t_map, 100*(1 - CFG["max_region_frac"])))

    r = max(1, int(CFG["open_frac"]*W))
    reg = binary_opening(reg, disk(r))
    reg = binary_closing(reg, disk(r))
    lab, n = ndi.label(reg)
    if n:
        sizes = ndi.sum(reg, lab, range(1, n+1))
        reg = np.isin(lab, 1 + np.nonzero(sizes >= CFG["min_region"]*sizes.max())[0])
    return reg

def split_arches_region(reg):
    """Pisah region gigi jadi lengkung atas & bawah lewat lembah profil baris."""
    row = reg.sum(1).astype(float)
    if row.max() == 0: return reg.copy(), np.zeros_like(reg)
    k = max(3, len(row)//40)
    sm = np.convolve(row, np.ones(k)/k, mode="same")
    lo, hi = int(.25*len(sm)), int(.80*len(sm))
    y = lo + int(np.argmin(sm[lo:hi])) if hi > lo else len(sm)//2
    up = reg.copy(); up[y:] = False
    dn = reg.copy(); dn[:y] = False
    return up, dn

## 2. Seed per gigi

Dua sumber bukti yang saling melengkapi:

**Distance transform.** Jarak tiap piksel ke latar. Tiap gigi punya "inti" — titik terjauh dari
tepinya. Gigi yang bersentuhan tetap punya dua puncak terpisah selama ada sedikit lekukan di
titik kontak. Puncak lokal dari distance transform = seed.

**Lembah interproksimal.** Distance transform bisa gagal saat dua gigi menyatu mulus (rotasi
berat, kontak lebar). Sebagai pelengkap: proyeksi vertikal kecerahan sepanjang lengkung membentuk
lembah di setiap sela gigi. Lembah yang tidak punya seed di antaranya → tambahkan seed.

Keduanya digabung, lalu seed yang terlalu berdekatan dilebur.

In [ ]:
def seed_spacing(dt, W):
    """
    Jarak minimum antar seed, diturunkan dari lebar gigi.

    Puncak distance transform di dalam satu gigi berjarak sekitar setengah lebarnya, dan
    dt.max() kira-kira setengah lebar gigi. Konstanta tetap (mis. 0.022 x lebar gambar)
    menghasilkan banyak seed per gigi begitu giginya besar — itulah sebabnya versi pertama
    memberi 24 seed untuk 6 gigi.
    """
    floor = max(3, int(CFG["seed_min_dist_frac"]*W))
    return max(floor, int(CFG["seed_dist_mult"]*float(dt.max()))) if dt.max() > 0 else floor

def seeds_from_dt(reg_arch, W):
    """Puncak lokal distance transform -> koordinat (x, y)."""
    if not reg_arch.any(): return np.zeros((0,2), int)
    dt = ndi.distance_transform_edt(reg_arch)
    pk = peak_local_max(dt, min_distance=seed_spacing(dt, W),
                        threshold_abs=CFG["seed_dt_frac"]*dt.max(), labels=reg_arch)
    return pk[:, ::-1] if len(pk) else np.zeros((0,2), int)   # -> (x, y)

def seeds_from_edges(rgb, reg_arch, W):
    """
    Seed dari region yang sudah DIPOTONG di garis tepi interproksimal.

    Ini penanganan untuk kasus tersulit: gigi yang tumpang tindih. Kalau dua siluet menyatu,
    distance transform-nya hanya punya satu punggungan dan menghasilkan satu seed untuk dua
    gigi — pada uji sintetis, tumpang tindih 20% cuma memberi 3 seed dari 6.

    Yang tetap ada meski siluetnya menyatu adalah **garis interproksimal**: batas enamel dan
    bayangan di antara dua gigi. Dengan membuang piksel bertepi kuat dari region sebelum
    menghitung distance transform, lekukan itu dibuat ulang secara artifisial dan tiap gigi
    kembali punya intinya sendiri.
    """
    if not reg_arch.any(): return np.zeros((0,2), int)
    t = toothness(rgb)
    gy, gx = np.gradient(t)
    edge = np.hypot(gx, gy)
    e_in = edge[reg_arch]
    if e_in.size < 20: return np.zeros((0,2), int)
    thr = float(np.percentile(e_in, CFG["edge_pct"]))
    cut = reg_arch & (edge < thr)
    cut = binary_opening(cut, disk(1))
    lab, n = ndi.label(cut)
    if n == 0: return np.zeros((0,2), int)

    # SATU seed per komponen, bukan peak_local_max dengan jarak minimum global.
    # Setelah dipotong di garis interproksimal, tiap inti gigi menjadi komponen tersendiri —
    # jadi mencacah komponen lebih tepat daripada mencari puncak. Jarak minimum global
    # justru merusak di sini: gumpalan gigi yang tumpang tindih itu tebal, sehingga
    # dt.max()-nya besar dan menekan puncak gigi tetangga.
    sizes = ndi.sum(cut, lab, range(1, n+1))
    med = float(np.median(sizes[sizes > 0])) if (sizes > 0).any() else 0.0
    out = []
    for i in range(1, n+1):
        if sizes[i-1] < max(CFG["min_cut_frac"]*med, 25): continue
        comp = lab == i
        d = ndi.distance_transform_edt(comp)
        yy, xx = np.unravel_index(int(np.argmax(d)), d.shape)
        out.append((int(xx), int(yy)))
    return np.array(out, int) if out else np.zeros((0,2), int)

def seeds_from_valleys(rgb, reg_arch, existing, W):
    """Tambah seed di antara lembah interproksimal yang belum terwakili."""
    if not reg_arch.any(): return np.zeros((0,2), int)
    _, val = sat_val(rgb)
    prof = np.where(reg_arch, val, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        col = np.nanmean(prof, 0)
    col = np.nan_to_num(col, nan=0.0)
    k = max(3, int(0.012*W))
    col = np.convolve(col, np.ones(k)/k, mode="same")
    rng = col.max() - col.min()
    if rng < 1e-6: return np.zeros((0,2), int)
    coln = (col - col.min())/rng
    dt_all = ndi.distance_transform_edt(reg_arch)
    sp = seed_spacing(dt_all, W)
    valleys, _ = find_peaks(-coln, prominence=CFG["valley_prominence"], distance=sp)
    bounds = [0] + list(valleys) + [W-1]
    out = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        if b - a < max(4, int(0.6*sp)): continue
        if any(a <= x <= b for x, _ in existing): continue
        band = reg_arch[:, a:b+1]
        if band.sum() < 30: continue
        dt = ndi.distance_transform_edt(band)
        yy, xx = np.unravel_index(np.argmax(dt), dt.shape)
        out.append((a + xx, yy))
    return np.array(out, int) if out else np.zeros((0,2), int)

def dedupe(pts, W, mind=None):
    mind = mind or max(3, int(CFG["seed_min_dist_frac"]*W))
    kept = []
    for p in pts:
        if all((p[0]-q[0])**2 + (p[1]-q[1])**2 > mind*mind for q in kept):
            kept.append(tuple(int(v) for v in p))
    return kept[:CFG["max_seeds"]]

def make_seeds(rgb):
    H, W = rgb.shape[:2]
    reg = tooth_region(rgb)
    up, dn = split_arches_region(reg)
    seeds = []
    for arch in (up, dn):
        s1 = seeds_from_dt(arch, W)                 # inti gigi
        s2 = seeds_from_edges(rgb, arch, W)         # gigi tumpang tindih, dipisah garis tepi
        have = [tuple(p) for p in s1] + [tuple(p) for p in s2]
        s3 = seeds_from_valleys(rgb, arch, have, W) # sela yang masih terlewat
        seeds += have + [tuple(p) for p in s3]
    # Peleburan seed kembar memakai jarak LEBIH KECIL daripada spacing pencarian puncak.
    # Gigi yang tumpang tindih punya pusat yang berdekatan; memakai spacing penuh di sini
    # akan melebur dua gigi asli menjadi satu seed.
    dt_all = ndi.distance_transform_edt(reg)
    mind = max(3, int(CFG["dedupe_mult"]*float(dt_all.max()))) if dt_all.max() > 0 else 3
    mind = max(mind, int(CFG["seed_min_dist_frac"]*W))
    return dedupe(seeds, W, mind), reg, up, dn

In [ ]:
# Lihat seed sebelum menyentuh SAM sama sekali — kalau seed-nya salah, mask pasti salah.
def preview_seeds(n_show=6):
    fig, axes = plt.subplots(n_show, 3, figsize=(13, 3.1*n_show))
    axes = np.atleast_2d(axes)
    for r in range(n_show):
        rgb = load_rgb(paths[r])
        seeds, reg, up, dn = make_seeds(rgb)
        axes[r,0].imshow(rgb); axes[r,0].set_title(names[r][:30], fontsize=8)
        axes[r,1].imshow(reg, cmap="gray"); axes[r,1].set_title("region gigi", fontsize=8)
        axes[r,2].imshow(rgb)
        if seeds:
            sx, sy = zip(*seeds)
            axes[r,2].plot(sx, sy, "o", ms=6, mfc="lime", mec="black", mew=.8)
        axes[r,2].set_title(f"{len(seeds)} seed", fontsize=8)
        for a in axes[r]: a.axis("off")
    plt.tight_layout(); plt.show()

preview_seeds(6)
print("Kalau seed sudah satu-per-gigi, sisanya tinggal SAM. Kalau belum, setel")
print("sat_pct/val_pct (region) atau seed_min_dist_frac/seed_dt_frac (seed) di CFG.")

## 3. SAM point-prompt

Tiap seed dikirim sebagai satu titik positif. `multimask_output=True` membuat SAM mengembalikan
tiga kandidat per titik (biasanya: bagian gigi, seluruh gigi, gigi + tetangga) — kita pilih yang
paling masuk akal sebagai satu gigi, bukan sekadar skor tertinggi.

Kriteria pemilihan: luas dalam rentang wajar, aspek mirip gigi, dan tumpang tindih tinggi dengan
region gigi (bukan menjalar ke gusi).

In [ ]:
from transformers import Sam2Model, Sam2Processor
model = Sam2Model.from_pretrained(CFG["model_id"]).to(DEV).eval()
processor = Sam2Processor.from_pretrained(CFG["model_id"])
print("dimuat:", CFG["model_id"])

@torch.no_grad()
def sam_points(rgb, seeds):
    """-> list of (mask bool, iou_score) per seed; kandidat terbaik sudah dipilih."""
    if not seeds: return []
    pil = Image.fromarray(rgb)
    H, W = rgb.shape[:2]
    out = []
    for i in range(0, len(seeds), CFG["sam_batch"]):
        chunk = seeds[i:i+CFG["sam_batch"]]
        pts = [[[list(map(int, p))] for p in chunk]]       # [batch][obj][point][xy]
        lbl = [[[1] for _ in chunk]]
        inp = processor(images=pil, input_points=pts, input_labels=lbl,
                        return_tensors="pt").to(DEV)
        o = model(**inp, multimask_output=CFG["multimask"])
        masks = processor.post_process_masks(
            o.pred_masks.cpu(), inp["original_sizes"].cpu())[0]      # (obj, cand, H, W)
        scores = o.iou_scores.cpu().numpy()[0]                        # (obj, cand)
        for j in range(masks.shape[0]):
            cands = [(np.asarray(masks[j, c], bool), float(scores[j, c]))
                     for c in range(masks.shape[1])]
            out.append(cands)
    return out

def pick_candidate(cands, reg, A):
    """Pilih kandidat yang paling menyerupai SATU gigi."""
    best, bs = None, -1e9
    for m, s in cands:
        a = m.sum()
        if a == 0: continue
        f = a/A
        if not (CFG["min_area_frac"] <= f <= CFG["max_area_frac"]): continue
        ys, xs = np.nonzero(m)
        bw, bh = xs.max()-xs.min()+1, ys.max()-ys.min()+1
        asp = bh/max(bw, 1)
        if not (CFG["aspect_min"] <= asp <= CFG["aspect_max"]): continue
        inside = np.logical_and(m, reg).sum()/a          # menempel di region gigi?
        score = 2.0*inside + s - 1.5*max(0.0, f/CFG["max_area_frac"] - 0.5)
        if score > bs: bs, best = score, (m, s)
    return best

## 4. Split & merge

Dua kesalahan khas yang tersisa setelah SAM:

**Satu mask memuat dua gigi.** Terjadi pada gigi berjejal yang bersentuhan rapat. Terdeteksi dari
lebar mask yang jauh melebihi median. Pemecahannya: watershed di dalam mask itu sendiri, dibimbing
distance transform-nya — dua inti gigi akan terpisah.

**Dua mask untuk satu gigi.** Terdeteksi dari IoU tinggi; yang lebih kecil dibuang.

In [ ]:
def split_wide(m, W):
    """Pecah mask yang kemungkinan memuat 2 gigi. -> list of mask."""
    dt = ndi.distance_transform_edt(m)
    pk = peak_local_max(dt, min_distance=seed_spacing(dt, W),
                        threshold_abs=0.45*dt.max(), labels=m)
    if len(pk) < 2: return [m]
    mk = np.zeros(m.shape, int)
    for i, (y, x) in enumerate(pk[:3], 1): mk[y, x] = i
    lab = watershed(-dt, mk, mask=m)
    parts = [lab == i for i in range(1, lab.max()+1)]
    return [p for p in parts if p.sum() > 0.15*m.sum()] or [m]

def postprocess(masks, rgb):
    H, W = rgb.shape[:2]; A = H*W
    if not masks: return []
    widths = []
    for m in masks:
        xs = np.nonzero(m)[1]
        widths.append(xs.max()-xs.min()+1)
    med = float(np.median(widths)) if widths else 0
    out = []
    for m, w in zip(masks, widths):
        if med > 0 and w > CFG["split_wide_ratio"]*med:
            out += split_wide(m, W)
        else:
            out.append(m)
    out.sort(key=lambda m: -m.sum())
    kept = []
    for m in out:
        f = m.sum()/A
        if not (CFG["min_area_frac"] <= f <= CFG["max_area_frac"]): continue
        dup = False
        for k in kept:
            inter = np.logical_and(m, k).sum()
            if inter and inter/min(m.sum(), k.sum()) > CFG["merge_iou"]: dup = True; break
        if not dup: kept.append(m)
    kept.sort(key=lambda m: np.nonzero(m)[1].mean())     # urut kiri -> kanan
    return kept

def segment_photo(rgb, seeds=None):
    if seeds is None:
        seeds, reg, _, _ = make_seeds(rgb)
    else:
        reg = tooth_region(rgb)
    A = rgb.shape[0]*rgb.shape[1]
    cand_sets = sam_points(rgb, seeds)
    raw = []
    for cands in cand_sets:
        pick = pick_candidate(cands, reg, A)
        if pick: raw.append(pick[0])
    return postprocess(raw, rgb), seeds, reg

In [ ]:
SEG = {}
t0 = time.time()
for p, n in zip(paths, names):
    try:
        rgb = load_rgb(p)
        masks, seeds, reg = segment_photo(rgb)
        SEG[n] = {"rgb": rgb, "masks": masks, "seeds": seeds, "reg": reg}
        cov = np.logical_or.reduce(masks).sum()/max(reg.sum(), 1) if masks else 0.0
        print(f"  {n[:42]:42s} {len(seeds):2d} seed -> {len(masks):2d} gigi   cakupan {cov*100:4.0f}%")
    except Exception as e:
        print(f"  {n[:42]:42s} GAGAL: {type(e).__name__} {str(e)[:70]}")
print(f"\n{len(SEG)} foto dalam {(time.time()-t0)/60:.1f} menit")

## 5. Quality gate

Tiga kriteria otomatis. Foto yang tidak lolos masuk daftar `NEEDS_FIX` dan diperbaiki di Bagian 7.

| Kriteria | Batas |
|---|---|
| Jumlah gigi | dalam `target_teeth` (default 8–20) |
| Cakupan | ≥ `min_coverage` dari region gigi tertutup mask |
| Celah besar | tidak ada wilayah region gigi tak-tertutup yang seukuran gigi |

In [ ]:
def qc(n):
    d = SEG[n]; masks, reg = d["masks"], d["reg"]
    A = d["rgb"].shape[0]*d["rgb"].shape[1]
    union = np.logical_or.reduce(masks) if masks else np.zeros_like(reg)
    cov = union.sum()/max(reg.sum(), 1)
    miss = np.logical_and(reg, ~union)
    lab, k = ndi.label(miss)
    med = float(np.median([m.sum() for m in masks])) if masks else 0
    big = 0
    if k and med > 0:
        sizes = ndi.sum(miss, lab, range(1, k+1))
        big = int((sizes > 0.55*med).sum())
    lo, hi = CFG["target_teeth"]
    ok = (lo <= len(masks) <= hi) and cov >= CFG["min_coverage"] and big == 0
    return {"foto": n, "n_gigi": len(masks), "n_seed": len(d["seeds"]),
            "cakupan": round(float(cov), 3), "celah_besar": big, "lolos": ok}

QC = pd.DataFrame([qc(n) for n in SEG])
print(QC.to_string(index=False))
NEEDS_FIX = QC[~QC.lolos].foto.tolist()
print(f"\nLOLOS {QC.lolos.sum()}/{len(QC)}   perlu koreksi: {len(NEEDS_FIX)}")
for n in NEEDS_FIX: print("   -", n)
QC.to_csv(os.path.join(CFG["out_dir"], "qc.csv"), index=False)

## 6. Tinjau visual

In [ ]:
def overlay(n, ax=None, show_seeds=True):
    d = SEG[n]; rgb, masks = d["rgb"], d["masks"]
    if ax is None: _, ax = plt.subplots(figsize=(7,5))
    ax.imshow(rgb); ax.axis("off")
    cmap = plt.get_cmap("tab20")
    for i, m in enumerate(masks):
        ax.contour(m, levels=[0.5], colors=[cmap(i % 20)], linewidths=1.7)
        ys, xs = np.nonzero(m)
        ax.text(xs.mean(), ys.mean(), str(i), fontsize=7, color="white", weight="bold",
                ha="center", va="center",
                bbox=dict(fc=cmap(i % 20), ec="none", alpha=.85, pad=.8))
    if show_seeds and d["seeds"]:
        sx, sy = zip(*d["seeds"]); ax.plot(sx, sy, "+", ms=7, mec="white", mew=1.2)
    r = QC[QC.foto == n].iloc[0]
    ax.set_title(f"{n[:34]}  {r.n_gigi} gigi  cov {r.cakupan:.2f}"
                 f"{'' if r.lolos else '  <-- PERLU KOREKSI'}", fontsize=8,
                 color=("black" if r.lolos else "crimson"))

ns = list(SEG); ncol = 3; nrow = int(np.ceil(len(ns)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.7*nrow))
for a in np.ravel(axes): a.axis("off")
for k, n in enumerate(ns): overlay(n, np.ravel(axes)[k])
plt.tight_layout(); plt.show()

## 7. Editor klik — koreksi manual

`%matplotlib widget` membuat gambar bisa diklik langsung di notebook.

- **Klik kiri** = tambah seed (gigi yang terlewat)
- **Klik kanan** = hapus seed terdekat (positif palsu / seed ganda)
- Jalankan sel berikutnya untuk **menjalankan ulang SAM** dengan seed hasil koreksi

Seed disimpan ke `seg_teeth/seeds_manual.json`, jadi koreksimu tidak hilang saat notebook
di-restart. Kalau `ipympl` tidak jalan, pakai sel cadangan di bawahnya (ketik koordinat manual).

In [ ]:
# Editor klik. Memeriksa ipympl LEBIH DULU: `%matplotlib widget` sbg magic telanjang
# selalu dieksekusi dan langsung RuntimeError kalau paketnya tidak ada.
import importlib.util
from IPython import get_ipython

HAVE_IPYMPL = importlib.util.find_spec("ipympl") is not None
if HAVE_IPYMPL:
    try:
        get_ipython().run_line_magic("matplotlib", "widget")
        print("ipympl aktif — editor klik siap.")
    except Exception as e:
        HAVE_IPYMPL = False
        print("ipympl ada tapi gagal diaktifkan:", type(e).__name__, str(e)[:100])
if not HAVE_IPYMPL:
    print("ipympl TIDAK tersedia. Dua pilihan:")
    print("  1) pip install ipympl   lalu RESTART KERNEL (magic backend dibaca saat start)")
    print("  2) lewati sel ini dan pakai Bagian 7b: grid koordinat + ketik seed manual")
    print("\nCatatan VS Code: ipympl kadang tetap rewel di sana; Jupyter Lab lebih andal.")

In [ ]:
import matplotlib.pyplot as plt

SEED_FILE = os.path.join(CFG["out_dir"], "seeds_manual.json")
MANUAL = json.load(open(SEED_FILE)) if os.path.exists(SEED_FILE) else {}

EDIT = NEEDS_FIX[0] if NEEDS_FIX else list(SEG)[0]   # ganti sesuai kebutuhan
print("Mengedit:", EDIT)

if not HAVE_IPYMPL:
    print("Lewati sel ini — pakai Bagian 7b.")
else:
    _cur = [list(map(list, MANUAL.get(EDIT, SEG[EDIT]["seeds"])))]
    _fig, _ax = plt.subplots(figsize=(9, 6.5))

    def _redraw():
        _ax.clear(); _ax.imshow(SEG[EDIT]["rgb"]); _ax.axis("off")
        for m in SEG[EDIT]["masks"]:
            _ax.contour(m, levels=[0.5], colors="cyan", linewidths=0.8, alpha=.55)
        if _cur[0]:
            sx, sy = zip(*_cur[0])
            _ax.plot(sx, sy, "o", ms=8, mfc="lime", mec="black", mew=1)
        _ax.set_title(f"{EDIT[:44]} — {len(_cur[0])} seed | kiri=tambah, kanan=hapus",
                      fontsize=9)
        _fig.canvas.draw_idle()

    def _onclick(ev):
        if ev.inaxes is not _ax or ev.xdata is None: return
        x, y = int(ev.xdata), int(ev.ydata)
        if ev.button == 1:
            _cur[0].append([x, y])
        elif ev.button == 3 and _cur[0]:
            d = [(px-x)**2 + (py-y)**2 for px, py in _cur[0]]
            _cur[0].pop(int(np.argmin(d)))
        MANUAL[EDIT] = _cur[0]
        with open(SEED_FILE, "w") as f: json.dump(MANUAL, f)
        _redraw()

    _fig.canvas.mpl_connect("button_press_event", _onclick)
    _redraw()

In [ ]:
# Jalankan ulang SAM pada seed hasil koreksi (foto yang sedang diedit saja)
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

pts = [tuple(p) for p in MANUAL.get(EDIT, SEG[EDIT]["seeds"])]
masks, seeds, reg = segment_photo(SEG[EDIT]["rgb"], seeds=pts)
SEG[EDIT].update({"masks": masks, "seeds": pts, "reg": reg})
QC = pd.DataFrame([qc(n) for n in SEG])
NEEDS_FIX = QC[~QC.lolos].foto.tolist()
print(QC[QC.foto == EDIT].to_string(index=False))
overlay(EDIT); plt.show()
print("sisa perlu koreksi:", NEEDS_FIX)

### 7b. Cadangan tanpa ipympl — grid koordinat

Selalu jalan, tanpa dependensi tambahan. Tampilkan foto bergaris koordinat, baca posisi tiap
gigi yang perlu seed, lalu ketik daftarnya di `MANUAL_SEEDS`.

In [ ]:
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def coord_grid(n, step=50):
    rgb = SEG[n]["rgb"]; H, W = rgb.shape[:2]
    fig, ax = plt.subplots(figsize=(12, 12*H/W))
    ax.imshow(rgb)
    for x in range(0, W, step): ax.axvline(x, color="w", lw=.4, alpha=.6)
    for y in range(0, H, step): ax.axhline(y, color="w", lw=.4, alpha=.6)
    ax.set_xticks(range(0, W, step*2)); ax.set_yticks(range(0, H, step*2))
    ax.tick_params(labelsize=7)
    for m in SEG[n]["masks"]:
        ax.contour(m, levels=[0.5], colors="cyan", linewidths=.8)
    if SEG[n]["seeds"]:
        sx, sy = zip(*SEG[n]["seeds"]); ax.plot(sx, sy, "o", ms=6, mfc="lime", mec="k")
    ax.set_title(f"{n[:50]} — baca koordinat, lalu isi MANUAL_SEEDS", fontsize=9)
    plt.tight_layout(); plt.show()

# coord_grid(NEEDS_FIX[0] if NEEDS_FIX else list(SEG)[0])

MANUAL_SEEDS = {
    # "nama file": [(x1,y1), (x2,y2), ...],
}
for n, pts in MANUAL_SEEDS.items():
    if n not in SEG: print("tidak ada:", n); continue
    masks, seeds, reg = segment_photo(SEG[n]["rgb"], seeds=[tuple(p) for p in pts])
    SEG[n].update({"masks": masks, "seeds": [tuple(p) for p in pts], "reg": reg})
    print(f"  {n[:44]:44s} -> {len(masks)} gigi")
if MANUAL_SEEDS:
    QC = pd.DataFrame([qc(n) for n in SEG]); NEEDS_FIX = QC[~QC.lolos].foto.tolist()
    print("\nsisa perlu koreksi:", NEEDS_FIX)

## 8. Uji akal sehat pada gigi berjejal sintetis

Menguji **implementasi** pemisahan, terpisah dari pertanyaan apakah SAM bekerja baik pada foto
asli. Tiga kondisi: gigi terpisah, gigi bersentuhan, dan gigi bertumpang tindih.

In [ ]:
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def synth(gap=0.0, n=6, W=560, H=340):
    """gap > 0 = ada celah antar gigi; gap < 0 = gigi TUMPANG TINDIH."""
    ys, xs = np.mgrid[0:H, 0:W]
    rgb = np.zeros((H, W, 3), np.uint8); rgb[:, :] = (198, 96, 108)
    widths = [58, 50, 66, 66, 50, 58][:n]
    x = 70.0; truth = []
    for w in widths:
        cx = x + w/2
        m = (((xs-cx)/((w-4)/2))**2 + ((ys-150)/60)**2) <= 1
        rim = (((xs-cx)/((w-4)/2))**2 + ((ys-150)/60)**2) <= 1.0
        inner = (((xs-cx)/((w-9)/2))**2 + ((ys-150)/57)**2) <= 1.0
        rgb[m] = (238, 214, 198)
        rgb[rim & ~inner] = (206, 182, 168)   # garis batas enamel spt foto asli
        truth.append(m)
        x += w*(1.0 + gap)
    return rgb, truth

print(f"{'kondisi':26s}{'seed':>6}{'target':>8}  hasil")
limit = None
for gp, lab in [(0.12, "renggang (celah 12%)"), (0.0, "bersentuhan"),
                (-0.10, "tumpang tindih 10%"), (-0.20, "tumpang tindih 20%"),
                (-0.30, "tumpang tindih 30%")]:
    rgb, truth = synth(gp)
    seeds, reg, up, dn = make_seeds(rgb)
    ok = len(seeds) == len(truth)
    if not ok and limit is None: limit = gp
    print(f"  {lab:24s}{len(seeds):6d}{len(truth):8d}  {'OK' if ok else 'kurang'}")
print(f"\nBATAS OPERASI OTOMATIS: sampai sekitar {abs(limit)*100:.0f}% tumpang tindih"
      if limit else "\nSemua kondisi tertangani otomatis")
print("Di atas itu siluet gigi menyatu dan garis interproksimalnya nyaris hilang —")
print("pakai editor klik (Bagian 7). Untuk n=18, itu jawaban yang benar, bukan kegagalan.")

# split_wide: satu mask memuat dua gigi -> harus terpecah jadi 2
ys, xs = np.mgrid[0:340, 0:560]
a = (((xs-200)/30)**2 + ((ys-150)/60)**2) <= 1
b = (((xs-252)/30)**2 + ((ys-150)/60)**2) <= 1
merged = np.logical_or(a, b)
parts = split_wide(merged, 560)
print(f"\nsplit_wide pada 2 gigi menyatu -> {len(parts)} bagian  "
      f"{'OK' if len(parts) == 2 else 'GAGAL'}")

In [ ]:
# Setel edge_pct pada FOTO ASLI, bukan sintetis. Sintetis hanya menguji implementasi;
# hanya foto asli yang bisa memberi tahu ambang tepi yang tepat untuk datamu.
def sweep_edge_pct(n, values=(60, 68, 74, 78, 84, 90)):
    rgb = SEG[n]["rgb"]
    base = CFG["edge_pct"]
    fig, axes = plt.subplots(1, len(values), figsize=(3.1*len(values), 3.4))
    for ax, v in zip(np.ravel(axes), values):
        CFG["edge_pct"] = v
        seeds, reg, up, dn = make_seeds(rgb)
        ax.imshow(rgb); ax.axis("off")
        if seeds:
            sx, sy = zip(*seeds); ax.plot(sx, sy, "o", ms=5, mfc="lime", mec="k", mew=.7)
        ax.set_title(f"edge_pct={v}\n{len(seeds)} seed", fontsize=8)
    CFG["edge_pct"] = base
    plt.suptitle(f"{n[:50]} — pilih nilai yang memberi satu seed per gigi", fontsize=10)
    plt.tight_layout(); plt.show()

_worst = QC.sort_values("n_gigi").foto.iloc[0]
sweep_edge_pct(_worst)
print(f"Foto: {_worst}")
print("Setel CFG['edge_pct'] ke nilai terbaik, lalu jalankan ulang sel segmentasi.")

## 9. Ekspor

In [ ]:
np.savez_compressed(os.path.join(CFG["out_dir"], "masks.npz"),
    **{f"{n}__{i}": SEG[n]["masks"][i] for n in SEG for i in range(len(SEG[n]["masks"]))})
with open(os.path.join(CFG["out_dir"], "seeds.json"), "w") as f:
    json.dump({n: [list(map(int, p)) for p in SEG[n]["seeds"]] for n in SEG}, f, indent=1)
QC.to_csv(os.path.join(CFG["out_dir"], "qc.csv"), index=False)

print(QC.to_string(index=False))
print(f"\nLOLOS {QC.lolos.sum()}/{len(QC)}  |  total gigi tersegmentasi: {QC.n_gigi.sum()}")
print("->", CFG["out_dir"], sorted(os.listdir(CFG["out_dir"])))

## Kalau masih ada yang meleset

Urutan penyetelan, dari yang paling berdampak:

1. **Region gigi salah** (Bagian 1 preview) — setel `sat_pct` naik / `val_pct` turun agar lebih
   permisif. Naikkan `col_tiles` kalau gigi pojok tetap hilang.
2. **Seed kurang** — turunkan `seed_dt_frac` dan `seed_min_dist_frac`; turunkan
   `valley_prominence` agar lembah interproksimal yang dangkal ikut terdeteksi.
3. **Seed berlebih** (dua seed di satu gigi) — naikkan `seed_min_dist_frac`.
4. **Mask menjalar ke gusi** — SAM memilih kandidat yang salah; turunkan `max_area_frac`.
5. **Dua gigi tetap menyatu** — turunkan `split_wide_ratio` ke ~1.35.

Kalau sebuah foto tetap bandel setelah semua itu, jangan buang waktu menyetel untuk satu kasus —
pakai editor klik. Untuk n=18, seed manual **adalah** jawaban yang benar, bukan kegagalan.